In [2]:
# from massive import RESTClient
# import json

# client = RESTClient("0_STHVfnT0CLigISYj9Oo0SIWVVpC9vO")

# # 先拉取少量数据看结构（只取前 5 条，避免持续翻页）
# balance_sheets = []
# for i, b in enumerate(
#     client.list_financials_balance_sheets(
#         limit=100,
#         sort="period_end.desc"
#         # 也可以加 tickers="AAPL" 进一步减少数据量
#     )
# ):
#     if i >= 5:
#         break

#     if hasattr(b, "model_dump"):
#         balance_sheets.append(b.model_dump())
#     elif hasattr(b, "__dict__"):
#         balance_sheets.append(b.__dict__)
#     else:
#         balance_sheets.append(dict(b))

# print(f"rows: {len(balance_sheets)}")
# if balance_sheets:
#     print("columns:", list(balance_sheets[0].keys()))
#     print("sample row:")
#     print(json.dumps(balance_sheets[0], ensure_ascii=False, indent=2, default=str))

In [3]:
# from massive import RESTClient
# import json
# import pandas as pd
# from pathlib import Path

# client = RESTClient("0_STHVfnT0CLigISYj9Oo0SIWVVpC9vO")

# # 先通过 REST API 拉取少量样本（只取前 5 条，避免持续翻页）
# balance_sheets = []
# for i, b in enumerate(
#     client.list_financials_balance_sheets(
#         limit=100,
#         sort="period_end.desc"
#     )
# ):
#     if i >= 5:
#         break

#     if hasattr(b, "model_dump"):
#         balance_sheets.append(b.model_dump())
#     elif hasattr(b, "__dict__"):
#         balance_sheets.append(b.__dict__)
#     else:
#         balance_sheets.append(dict(b))

# # 将样本数据落盘为 parquet
# df = pd.DataFrame(balance_sheets)
# out_path = Path("balance_sheets_sample.parquet")
# df.to_parquet(out_path, index=False)

# # 再读回来展示
# df_read = pd.read_parquet(out_path)
# print(f"saved: {out_path.resolve()}")
# print(f"shape: {df_read.shape}")
# display(df_read.head())

In [4]:
# import math
# import time

# # 统计参数
# limit = 50000  # 每页最大 50000
# max_rows_to_scan = None  # 先扫描 20 万条做快速估算；设为 None 可做全量精确统计

# rows = 0
# start = time.time()

# for rows, _ in enumerate(
#     client.list_financials_balance_sheets(
#         limit=limit,
#         sort="period_end.desc"
#     ),
#     start=1
# ):
#     if rows % 50000 == 0:
#         print(f"scanned rows: {rows}")
#     if max_rows_to_scan is not None and rows >= max_rows_to_scan:
#         break

# elapsed = time.time() - start
# pages_scanned = math.ceil(rows / limit)

# print(f"elapsed: {elapsed:.2f}s")
# print(f"rows_scanned: {rows}")
# print(f"pages_scanned: {pages_scanned}")

# if max_rows_to_scan is None:
#     print(f"exact_total_rows: {rows}")
#     print(f"exact_total_pages(@limit={limit}): {math.ceil(rows / limit)}")
# else:
#     print("estimated_total_rows: >= rows_scanned (lower bound)")
#     print(f"estimated_total_pages(@limit={limit}): >= {pages_scanned}")
#     print("要拿精确总数：把 max_rows_to_scan 改成 None 再运行一次。")

In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urlparse, parse_qs
import json
import random
import time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from massive import RESTClient

class MassiveRestDownloader:
    """Unified downloader for Massive datasets across doc categories."""

    DATASETS = {
        # fundamentals
        "balance_sheet": {"category": "fundamentals", "source": "sdk", "m": "list_financials_balance_sheets", "sub": "balance_sheet", "p": "balance_sheet", "mode": "fiscal", "sort": "period_end.asc"},
        "income_statement": {"category": "fundamentals", "source": "sdk", "m": "list_financials_income_statements", "sub": "income_statement", "p": "income_statement", "mode": "fiscal", "sort": "period_end.asc"},
        "cash_flow_statement": {"category": "fundamentals", "source": "sdk", "m": "list_financials_cash_flow_statements", "sub": "cash_flow_statement", "p": "cash_flow_statement", "mode": "fiscal", "sort": "period_end.asc"},
        "financials_ratios": {"category": "fundamentals", "source": "sdk", "m": "list_financials_ratios", "sub": "financials_ratios", "p": "financials_ratios", "mode": "all", "sort": "ticker.asc"},
        "stocks_floats": {"category": "fundamentals", "source": "raw", "endpoint": "/stocks/vX/float", "sub": "stocks_floats", "p": "stocks_floats", "mode": "all", "sort": "ticker.asc"},
        "short_interest": {"category": "fundamentals", "source": "sdk", "m": "list_short_interest", "sub": "short_interest", "p": "short_interest", "mode": "date", "date_fields": ["settlement_date", "date", "as_of_date"], "sort": "settlement_date.asc,ticker.asc"},
        "short_volume": {"category": "fundamentals", "source": "sdk", "m": "list_short_volume", "sub": "short_volume", "p": "short_volume", "mode": "date", "date_fields": ["date", "settlement_date", "as_of_date"], "sort": "date.asc,ticker.asc"},

        # aggregate_bars (remove previous_day_bar)
        "aggs_daily_market_summary": {"category": "aggregate_bars", "source": "raw", "endpoint_template": "/v2/aggs/grouped/locale/us/market/stocks/{date}", "sub": "daily_market_summary", "p": "daily_market_summary", "mode": "calendar_year", "start_year": 2004, "page_pause_s": 0.05, "include_otc": "false", "adjusted": "true"},

        # filing
        "filing_sec_edgar_index": {"category": "filing", "source": "raw", "endpoint": "/stocks/filings/vX/index", "sub": "sec_edgar_index", "p": "sec_edgar_index", "mode": "all", "sort": "filing_date.desc"},
        "filing_8k_text": {"category": "filing", "source": "raw", "endpoint": "/stocks/filings/8-K/vX/text", "sub": "8k_text", "p": "8k_text", "mode": "date", "partition_freq": "month", "date_fields": ["filing_date"], "sort": "filing_date.desc", "page_pause_s": 0.25},
        "filing_10k_sections": {"category": "filing", "source": "raw", "endpoint": "/stocks/filings/10-K/vX/sections", "sub": "10k_sections", "p": "10k_sections", "mode": "all", "sort": "period_end.desc"},
        "filing_risk_categories": {"category": "filing", "source": "raw", "endpoint": "/stocks/taxonomies/vX/risk-factors", "sub": "risk_categories", "p": "risk_categories", "mode": "all", "sort": "taxonomy.desc"},
        "filing_risk_factors": {"category": "filing", "source": "auto", "m": "list_stocks_filings_risk_factors", "endpoint": "/stocks/filings/vX/risk-factors", "sub": "risk_factors", "p": "risk_factors", "mode": "date", "date_fields": ["filing_date"], "sort": "filing_date.desc"},

        # corporate_actions
        "corp_dividends": {"category": "corporate_actions", "source": "raw", "endpoint": "/stocks/v1/dividends", "sub": "dividends", "p": "dividends", "mode": "date", "date_fields": ["ex_dividend_date", "pay_date", "declaration_date"], "sort": "ex_dividend_date.desc"},
        "corp_splits": {"category": "corporate_actions", "source": "raw", "endpoint": "/stocks/v1/splits", "sub": "splits", "p": "splits", "mode": "date", "date_fields": ["execution_date"], "sort": "execution_date.desc"},
        "corp_ipos": {"category": "corporate_actions", "source": "raw", "endpoint": "/vX/reference/ipos", "sub": "ipos", "p": "ipos", "mode": "all", "max_limit": 1000},

        # news
        "news_all": {"category": "news", "source": "raw", "endpoint": "/v2/reference/news", "sub": "news", "p": "news", "mode": "all"},

        # market_operations (remove market_status)
        "market_holidays": {"category": "market_operations", "source": "raw", "endpoint": "/v1/marketstatus/upcoming", "sub": "market_holidays", "p": "market_holidays", "mode": "all", "no_limit": True, "sort": "date.asc"},
        "market_exchanges": {"category": "market_operations", "source": "raw", "endpoint": "/v3/reference/exchanges", "sub": "exchanges", "p": "exchanges", "mode": "all", "no_limit": True, "sort": "name.asc"},
        "market_condition_codes": {"category": "market_operations", "source": "raw", "endpoint": "/v3/reference/conditions", "sub": "condition_codes", "p": "condition_codes", "mode": "all", "no_limit": True},

        # tickers
        "tickers_all": {"category": "tickers", "source": "raw", "endpoint": "/v3/reference/tickers", "sub": "all_tickers", "p": "all_tickers", "mode": "all", "max_limit": 1000},
        "tickers_types": {"category": "tickers", "source": "raw", "endpoint": "/v3/reference/tickers/types", "sub": "ticker_types", "p": "ticker_types", "mode": "all", "no_limit": True, "sort": "code.asc"},
    }

    def __init__(self, client, root_dir="/home/yluel/share/projects/massive_parquet", request_max_retries=12, request_backoff_base=1.7, request_backoff_cap=90.0, request_jitter=0.6, cursor_retry_floor_s=6.0):
        self.client = client
        self.root = Path(root_dir)
        self.root.mkdir(parents=True, exist_ok=True)
        self.request_max_retries = int(request_max_retries)
        self.request_backoff_base = float(request_backoff_base)
        self.request_backoff_cap = float(request_backoff_cap)
        self.request_jitter = float(request_jitter)
        self.cursor_retry_floor_s = float(cursor_retry_floor_s)

    def _marker_path(self, fp):
        return Path(str(fp) + ".ok")

    def _is_retryable_error(self, exc):
        s = str(exc).lower()
        retry_tokens = ["502", "503", "504", "429", "too many", "timed out", "timeout", "temporarily unavailable", "connection reset", "max retries exceeded"]
        return any(t in s for t in retry_tokens)

    def _raw_get_with_retry(self, path, params):
        attempt = 0
        while True:
            try:
                return self.client._get(path=path, params=params, raw=True)
            except Exception as exc:
                attempt += 1
                if (not self._is_retryable_error(exc)) or (attempt > self.request_max_retries):
                    raise
                sleep_s = min(self.request_backoff_cap, self.request_backoff_base ** attempt + random.uniform(0.0, self.request_jitter))
                if params and params.get("cursor"):
                    sleep_s = max(sleep_s, self.cursor_retry_floor_s + random.uniform(0.0, 1.0))
                print(f"[retry] path={path} attempt={attempt}/{self.request_max_retries} sleep={sleep_s:.1f}s err={exc}")
                time.sleep(sleep_s)

    def _iter_raw(self, endpoint, max_pages=None, page_pause_s=0.0, **kwargs):
        path = endpoint
        params = dict(kwargs)
        page = 0
        while True:
            resp = self._raw_get_with_retry(path=path, params=params)
            payload = json.loads(resp.data.decode("utf-8"))
            page += 1
            if isinstance(payload, list):
                rows = payload
            else:
                rows = payload.get("results", [])
            if isinstance(rows, dict):
                rows = [rows]
            for row in rows:
                yield row
            if max_pages is not None and page >= int(max_pages):
                break
            next_url = payload.get("next_url") if isinstance(payload, dict) else None
            if not next_url:
                break
            if page_pause_s and float(page_pause_s) > 0:
                time.sleep(float(page_pause_s))
            parsed = urlparse(next_url)
            path = parsed.path
            params = {k: v[0] for k, v in parse_qs(parsed.query).items()}

    def _iter(self, ds, max_pages=None, **kwargs):
        cfg = self.DATASETS[ds]
        source = cfg.get("source")
        if source == "raw":
            return self._iter_raw(cfg["endpoint"], max_pages=max_pages, page_pause_s=cfg.get("page_pause_s", 0.0), **kwargs)
        method_name = cfg.get("m")
        if source == "sdk":
            if method_name and hasattr(self.client, method_name):
                return getattr(self.client, method_name)(**kwargs)
            raise AttributeError(f"RESTClient has no method: {method_name}")
        if source == "auto":
            if method_name and hasattr(self.client, method_name):
                return getattr(self.client, method_name)(**kwargs)
            return self._iter_raw(cfg["endpoint"], max_pages=max_pages, page_pause_s=cfg.get("page_pause_s", 0.0), **kwargs)
        raise ValueError(f"Unknown source for dataset {ds}: {source}")

    def _to_dict(self, x):
        return x.model_dump() if hasattr(x, "model_dump") else (x.__dict__ if hasattr(x, "__dict__") else dict(x))

    def _detect_date_field(self, ds):
        fields = self.DATASETS[ds].get("date_fields", [])
        first = next(self._iter(ds, limit=1, max_pages=1), None)
        if first is None:
            raise RuntimeError(f"No data returned for {ds}")
        rec = self._to_dict(first)
        for f in fields:
            if f in rec and rec.get(f) is not None:
                return f
        raise RuntimeError(f"Cannot find valid date field for {ds}. Candidates={fields}")

    def _partitions(self, ds):
        c = self.DATASETS[ds]
        if c["mode"] == "all":
            return ["all"], None
        if c["mode"] == "calendar_year":
            start_y = int(c.get("start_year", 2004))
            end_y = int(pd.Timestamp.today().year)
            return list(range(start_y, end_y + 1)), None
        if c["mode"] == "fiscal":
            a = self._to_dict(next(self._iter(ds, limit=1, sort="fiscal_year.asc,period_end.asc", max_pages=1), None))
            b = self._to_dict(next(self._iter(ds, limit=1, sort="fiscal_year.desc,period_end.desc", max_pages=1), None))
            return list(range(int(float(a["fiscal_year"])), int(float(b["fiscal_year"])) + 1)), None

        date_field = self._detect_date_field(ds)
        a = self._to_dict(next(self._iter(ds, limit=1, sort=f"{date_field}.asc", max_pages=1), None))
        b = self._to_dict(next(self._iter(ds, limit=1, sort=f"{date_field}.desc", max_pages=1), None))
        freq = c.get("partition_freq", "year")
        if freq == "month":
            start_month = pd.to_datetime(a[date_field]).to_period("M")
            end_month = pd.to_datetime(b[date_field]).to_period("M")
            months = pd.period_range(start=start_month, end=end_month, freq="M")
            return [str(m) for m in months], date_field
        ya = pd.to_datetime(a[date_field]).year
        yb = pd.to_datetime(b[date_field]).year
        return list(range(int(ya), int(yb) + 1)), date_field

    def _query(self, ds, part, limit, sort, date_field):
        c = self.DATASETS[ds]
        eff_limit = None if c.get("no_limit", False) else int(min(int(limit), int(c.get("max_limit", limit))))
        q = {"sort": sort or c.get("sort", "")}
        if eff_limit is not None:
            q["limit"] = eff_limit
        if c["mode"] == "fiscal":
            q["fiscal_year"] = int(part)
        elif c["mode"] == "date" and part != "all" and date_field is not None:
            if isinstance(part, str) and len(part) == 7 and part[4] == "-":
                start_ts = pd.Period(part, freq="M").start_time
                end_ts = (pd.Period(part, freq="M") + 1).start_time
                start_s = start_ts.strftime("%Y-%m-%d")
                end_s = end_ts.strftime("%Y-%m-%d")
            else:
                start_s = f"{part}-01-01"
                end_s = f"{int(part) + 1}-01-01"
            if c.get("source") == "raw":
                q[f"{date_field}.gte"] = start_s
                q[f"{date_field}.lt"] = end_s
            else:
                q[f"{date_field}_gte"] = start_s
                q[f"{date_field}_lt"] = end_s
        return {k: v for k, v in q.items() if v not in ("", None)}

    def _stream_write_df_chunks(self, fp, records_iter, ds, chunk_size=20000):
        rows = 0
        chunk = []
        writer = None
        base_cols = None
        if fp.exists():
            fp.unlink()

        def flush_chunk(records):
            nonlocal rows, writer, base_cols
            if not records:
                return
            df = pd.DataFrame(records)
            # Normalize known drifting columns to keep parquet schema stable across pages.
            if ds == "corp_dividends" and "declaration_date" in df.columns:
                df["declaration_date"] = df["declaration_date"].fillna("").astype(str)
            if ds == "filing_risk_factors" and "ticker" in df.columns:
                df["ticker"] = df["ticker"].fillna("").astype(str)
            if ds == "income_statement" and "extraordinary_items" in df.columns:
                df["extraordinary_items"] = pd.to_numeric(df["extraordinary_items"], errors="coerce")
            if ds == "income_statement" and "equity_in_affiliates" in df.columns:
                df["equity_in_affiliates"] = pd.to_numeric(df["equity_in_affiliates"], errors="coerce")
            if ds == "tickers_all":
                for c_name in ["currency_symbol", "base_currency_symbol", "base_currency_name"]:
                    if c_name in df.columns:
                        df[c_name] = df[c_name].fillna("").astype(str)
            if ds == "cash_flow_statement":
                for c_name in [
                    "income_loss_from_discontinued_operations",
                    "net_cash_from_financing_activities_discontinued_operations",
                    "net_cash_from_investing_activities_discontinued_operations",
                    "net_cash_from_operating_activities_discontinued_operations",
                    "other_cash_adjustments",
                ]:
                    if c_name in df.columns:
                        df[c_name] = pd.to_numeric(df[c_name], errors="coerce")
            if ds == "aggs_daily_market_summary" and "n" in df.columns:
                df["n"] = pd.to_numeric(df["n"], errors="coerce").astype("float64")
            if base_cols is None:
                base_cols = list(df.columns)
            else:
                for c_name in base_cols:
                    if c_name not in df.columns:
                        df[c_name] = None
                extra_cols = [x for x in df.columns if x not in base_cols]
                if extra_cols:
                    df = df.drop(columns=extra_cols)
                df = df.reindex(columns=base_cols)
            table = pa.Table.from_pandas(df, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(str(fp), table.schema, compression="snappy")
            writer.write_table(table)
            rows += len(df)

        try:
            for rec in records_iter:
                chunk.append(self._to_dict(rec))
                if len(chunk) >= int(chunk_size):
                    flush_chunk(chunk)
                    chunk = []
            if chunk:
                flush_chunk(chunk)
        finally:
            if writer is not None:
                writer.close()
        return rows

    def _stream_write_calendar_year(self, ds, year, fp, chunk_size=20000):
        cfg = self.DATASETS[ds]
        year_i = int(year)
        start = pd.Timestamp(year=year_i, month=1, day=1)
        end = pd.Timestamp.today().normalize() if year_i == int(pd.Timestamp.today().year) else pd.Timestamp(year=year_i, month=12, day=31)
        biz_days = pd.bdate_range(start=start, end=end)

        def records():
            pbar = tqdm(biz_days, desc=f"{ds}:{year_i}", unit="day", leave=False)
            for d in pbar:
                date_s = d.strftime("%Y-%m-%d")
                path = cfg["endpoint_template"].format(date=date_s)
                params = {"adjusted": cfg.get("adjusted", "true"), "include_otc": cfg.get("include_otc", "false")}
                payload = json.loads(self._raw_get_with_retry(path=path, params=params).data.decode("utf-8"))
                for rec in payload.get("results", []) or []:
                    row = rec if isinstance(rec, dict) else self._to_dict(rec)
                    row["trade_date"] = date_s
                    yield row
                pause_s = float(cfg.get("page_pause_s", 0.0))
                if pause_s > 0:
                    time.sleep(pause_s)

        return self._stream_write_df_chunks(fp, records(), ds=ds, chunk_size=chunk_size)

    def _stream_write_partition(self, ds, query, fp, chunk_size=20000, max_pages=None):
        cfg = self.DATASETS[ds]
        if cfg.get("source") == "sdk":
            max_rows = None
            if max_pages is not None:
                max_rows = int(max_pages) * int(query.get("limit", 0))
            def sdk_records():
                rows_seen = 0
                for rec in self._iter(ds, **query):
                    yield rec
                    rows_seen += 1
                    if max_rows is not None and rows_seen >= max_rows:
                        break
            return self._stream_write_df_chunks(fp, sdk_records(), ds=ds, chunk_size=chunk_size)
        return self._stream_write_df_chunks(fp, self._iter(ds, max_pages=max_pages, **query), ds=ds, chunk_size=chunk_size)

    def download_dataset(self, ds, years=None, workers=8, limit=50000, sort=None, skip_existing=True, delete_empty=True, max_pages=None, chunk_size=20000):
        c = self.DATASETS[ds]
        out = self.root / c["category"] / c["sub"]
        out.mkdir(parents=True, exist_ok=True)

        parts, date_field = self._partitions(ds)
        if years is not None and c["mode"] in ("fiscal", "date", "calendar_year") and parts != ["all"]:
            parts = sorted(list(years))

        def one(part):
            suffix = part if part != "all" else "all"
            fp = out / f"{c['p']}_{suffix}.parquet"
            marker_fp = self._marker_path(fp)
            if skip_existing and marker_fp.exists():
                return {"dataset": ds, "category": c["category"], "partition": suffix, "rows": None, "status": "skipped", "file": str(fp), "marker": str(marker_fp)}
            if marker_fp.exists():
                marker_fp.unlink()

            if c["mode"] == "calendar_year":
                rows = self._stream_write_calendar_year(ds=ds, year=part, fp=fp, chunk_size=chunk_size)
            else:
                query = self._query(ds, part, limit, sort, date_field)
                rows = self._stream_write_partition(ds=ds, query=query, fp=fp, chunk_size=chunk_size, max_pages=max_pages)

            if rows == 0:
                if delete_empty and fp.exists():
                    fp.unlink()
                if marker_fp.exists():
                    marker_fp.unlink()
                return {"dataset": ds, "category": c["category"], "partition": suffix, "rows": 0, "status": "empty", "file": str(fp), "marker": str(marker_fp)}

            marker_payload = {"dataset": ds, "partition": str(suffix), "rows": int(rows), "updated_at": pd.Timestamp.utcnow().isoformat()}
            marker_fp.write_text(json.dumps(marker_payload, ensure_ascii=False), encoding="utf-8")
            return {"dataset": ds, "category": c["category"], "partition": suffix, "rows": rows, "status": "ok", "file": str(fp), "marker": str(marker_fp)}

        res = []
        with ThreadPoolExecutor(max_workers=workers) as ex:
            fs = [ex.submit(one, p) for p in parts]
            pbar = tqdm(total=len(fs), desc=f"{c['category']}:{ds}", unit="part")
            for f in as_completed(fs):
                x = f.result()
                res.append(x)
                pbar.update(1)
                print(f"[{x['category']}][{ds}][{x['partition']}] {x['status']} rows={x['rows']}")
            pbar.close()
        return pd.DataFrame(res).sort_values("partition").reset_index(drop=True)

    def download_all(self, datasets=None, years=None, dataset_workers=4, partition_workers=8, limit=50000, sort=None, skip_existing=True, delete_empty=True, max_pages=None, chunk_size=20000):
        dss = datasets or list(self.DATASETS.keys())
        out = {}
        with ThreadPoolExecutor(max_workers=dataset_workers) as ex:
            mp = {
                ex.submit(
                    self.download_dataset,
                    ds,
                    years,
                    partition_workers,
                    limit,
                    sort,
                    skip_existing,
                    delete_empty,
                    max_pages,
                    chunk_size,
                ): ds
                for ds in dss
            }
            for f in as_completed(mp):
                ds = mp[f]
                try:
                    out[ds] = f.result()
                except Exception as e:
                    cfg = self.DATASETS[ds]
                    out[ds] = pd.DataFrame([{"dataset": ds, "category": cfg["category"], "partition": "all", "rows": None, "status": "failed", "error": str(e)}])
                    print(f"[{cfg['category']}][{ds}] failed: {e}")

        all_summary = pd.concat([out[k] for k in out], ignore_index=True)
        return out, all_summary

client = RESTClient("0_STHVfnT0CLigISYj9Oo0SIWVVpC9vO")
downloader = MassiveRestDownloader(
    client,
    root_dir="/home/yluel/share/projects/massive_parquet",
    request_max_retries=12,
    request_backoff_base=1.7,
    request_backoff_cap=90.0,
    request_jitter=0.6,
    cursor_retry_floor_s=6.0,
)

# 去掉 snapshots / market_status / previous_day_bar + 8k/10k 后可直接批量下载的数据类型
datasets_to_download = [
    "balance_sheet", "income_statement", "cash_flow_statement", "financials_ratios",
    "stocks_floats", "short_interest", "short_volume",
    "aggs_daily_market_summary",
    "filing_sec_edgar_index", "filing_risk_categories", "filing_risk_factors",
    "corp_dividends", "corp_splits", "corp_ipos",
    "news_all",
    "market_holidays", "market_exchanges", "market_condition_codes",
    "tickers_all", "tickers_types",
]

results_by_dataset, all_summary = downloader.download_all(
    datasets=datasets_to_download,
    years=None,
    dataset_workers=10,
    partition_workers=50,
    limit=5000,
    skip_existing=True,
    delete_empty=True,
    max_pages=None,
    chunk_size=10000,
)
all_summary.sort_values(["category", "dataset", "partition"]).reset_index(drop=True).to_csv("download_results.csv")
display(all_summary.sort_values(["category", "dataset", "partition"]).reset_index(drop=True))

filing:filing_sec_edgar_index:   0%|          | 0/1 [00:00<?, ?part/s]

aggregate_bars:aggs_daily_market_summary:   0%|          | 0/23 [00:00<?, ?part/s]

fundamentals:financials_ratios:   0%|          | 0/1 [00:00<?, ?part/s]

fundamentals:stocks_floats:   0%|          | 0/1 [00:00<?, ?part/s]

filing:filing_risk_categories:   0%|          | 0/1 [00:00<?, ?part/s]

...

[fundamentals][short_interest][2018] skipped rows=None
[fundamentals][short_interest][2024] skipped rows=None
[fundamentals][short_interest][2019] skipped rows=None
[fundamentals][short_interest][2025] skipped rows=None
[fundamentals][short_interest][2022] skipped rows=None
[fundamentals][short_interest][2021] skipped rows=None
[fundamentals][short_interest][2020] skipped rows=None
[fundamentals][short_interest][2023] skipped rows=None
[fundamentals][short_interest][2026] skipped rows=None
[fundamentals][short_interest][2017] skipped rows=None
[filing][filing_risk_factors][2024] skipped rows=None
[filing][filing_risk_factors][2017] skipped rows=None
[filing][filing_risk_factors][2026] skipped rows=None
[filing][filing_risk_factors][2020] skipped rows=None
[filing][filing_risk_factors][2015] skipped rows=None
[filing][filing_risk_factors][2025] skipped rows=None
[filing][filing_risk_factors][2022] skipped rows=None
[filing][filing_risk_factors][2019] skipped rows=None
[filing][filing_ri

fundamentals:short_volume:   0%|          | 0/3 [00:00<?, ?part/s]

[fundamentals][short_volume][2024] skipped rows=None
[fundamentals][short_volume][2026] skipped rows=None
[fundamentals][short_volume][2025] skipped rows=None


/tmp/ipykernel_1724796/3902180428.py:377: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_summary = pd.concat([out[k] for k in out], ignore_index=True)


,dataset,category,partition,rows,status,file,marker
0,aggs_daily_market_summary,aggregate_bars,2004,NaN,skipped,/home/yluel/share/projects/massive_parquet/agg...,/home/yluel/share/projects/massive_parquet/agg...
1,aggs_daily_market_summary,aggregate_bars,2005,NaN,skipped,/home/yluel/share/projects/massive_parquet/agg...,/home/yluel/share/projects/massive_parquet/agg...
2,aggs_daily_market_summary,aggregate_bars,2006,NaN,skipped,/home/yluel/share/projects/massive_parquet/agg...,/home/yluel/share/projects/massive_parquet/agg...
3,aggs_daily_market_summary,aggregate_bars,2007,NaN,skipped,/home/yluel/share/projects/massive_parquet/agg...,/home/yluel/share/projects/massive_parquet/agg...
4,aggs_daily_market_summary,aggregate_bars,2008,NaN,skipped,/home/yluel/share/projects/massive_parquet/agg...,/home/yluel/share/projects/massive_parquet/agg...
...,...,...,...,...,...,...,...
187,market_exchanges,market_operations,all,NaN,skipped,/home/yluel/share/projects/massive_parquet/mar...,/home/yluel/share/projects/massive_parquet/mar...
188,market_holidays,market_operations,all,NaN,skipped,/home/yluel/share/projects/massive_parquet/mar...,/home/yluel/share/projects/massive_parquet/mar...
189,news_all,news,all,NaN,skipped,/home/yluel/share/projects/massive_parquet/new...,/home/yluel/share/projects/massive_parquet/new...
190,tickers_all,tickers,all,NaN,skipped,/home/yluel/share/projects/massive_parquet/tic...,/home/yluel/share/projects/massive_parquet/tic...


In [6]:
# 冒烟测试：验证 filing/sec_edgar_index 可落盘（限制页数，防止跑太久）
smoke_summary = downloader.download_dataset(
    ds="filing_sec_edgar_index",
    workers=1,
    limit=1000,
    skip_existing=True,
    delete_empty=True,
    max_pages=3,
    chunk_size=5000,
 )
display(smoke_summary.head())

[filing][filing_sec_edgar_index][all] skipped rows=None


,dataset,category,partition,rows,status,file
0,filing_sec_edgar_index,filing,all,None,skipped,/home/yluel/share/projects/massive_parquet/fil...


In [ ]:
# 全量下载（aggregate_bars + filing，10-K 先跳过）
full_datasets = [
    "aggs_daily_market_summary",
    "filing_sec_edgar_index",
    "filing_8k_text",
    # "filing_10k_sections",  # 暂时跳过
    "filing_risk_categories",
    "filing_risk_factors",
 ]

full_results, full_summary = downloader.download_all(
    datasets=full_datasets,
    years=None,                 # aggs_daily_market_summary 为按年分片；8-K 按月分片
    dataset_workers=1,          # 稳态优先，避免网关波动放大
    partition_workers=1,
    limit=1000,                 # 对 filing/raw 保守分页
    skip_existing=False,        # 启用全量下载（覆盖已有文件）
    delete_empty=True,
    max_pages=None,             # None = 全量
    chunk_size=10000,
 )

display(
    full_summary.sort_values(["category", "dataset", "partition"]).reset_index(drop=True)
 )

[retry] path=/stocks/filings/8-K/vX/text attempt=1/12 sleep=6.0s err=HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /stocks/filings/8-K/vX/text?cursor=AggJ-xaaCQn7VJoBBAAAAfvoAwABAQn7QJo%3D (Caused by ResponseError('too many 502 error responses'))


In [ ]:
# 轻量重试测试：仅验证之前失败的两个 filing 数据集
retry_probe_datasets = ["filing_8k_text", "filing_10k_sections"]
probe_results, probe_summary = downloader.download_all(
    datasets=retry_probe_datasets,
    years=None,
    dataset_workers=1,
    partition_workers=1,
    limit=1000,
    skip_existing=False,
    delete_empty=True,
    max_pages=1,
    chunk_size=2000,
 )
display(probe_summary.sort_values(["dataset", "partition"]).reset_index(drop=True))

[filing][filing_8k_text][2024] ok rows=1000
[filing][filing_8k_text][2025] ok rows=1000
[filing][filing_8k_text][2026] ok rows=1000


In [ ]:
# 轻量测试：filing_risk_factors
rf_summary = downloader.download_dataset(
    ds="filing_risk_factors",
    workers=1,
    limit=1000,
    skip_existing=False,
    delete_empty=True,
    max_pages=1,
    chunk_size=2000,
 )
display(rf_summary.sort_values(["partition"]).reset_index(drop=True).head())

[filing][filing_risk_factors][2015] ok rows=1000
[filing][filing_risk_factors][2016] ok rows=1000
[filing][filing_risk_factors][2017] ok rows=1000
[filing][filing_risk_factors][2018] ok rows=1000
[filing][filing_risk_factors][2019] ok rows=1000
[filing][filing_risk_factors][2020] ok rows=1000
[filing][filing_risk_factors][2021] ok rows=1000
[filing][filing_risk_factors][2022] ok rows=1000
[filing][filing_risk_factors][2023] ok rows=1000
[filing][filing_risk_factors][2024] ok rows=1000
[filing][filing_risk_factors][2025] ok rows=1000
[filing][filing_risk_factors][2026] ok rows=1000


,dataset,category,partition,rows,status,file
0,filing_risk_factors,filing,2015,1000,ok,/home/yluel/share/projects/massive_parquet/fil...
1,filing_risk_factors,filing,2016,1000,ok,/home/yluel/share/projects/massive_parquet/fil...
2,filing_risk_factors,filing,2017,1000,ok,/home/yluel/share/projects/massive_parquet/fil...
3,filing_risk_factors,filing,2018,1000,ok,/home/yluel/share/projects/massive_parquet/fil...
4,filing_risk_factors,filing,2019,1000,ok,/home/yluel/share/projects/massive_parquet/fil...


In [ ]:
# 探测 SDK 可用的方法（含 filings/taxonomies/short/financials）
method_names = [n for n in dir(client) if n.startswith("list_")]
for n in sorted(method_names):
    if any(k in n.lower() for k in ["filing", "tax", "risk", "short", "financials"]):
        print(n)

list_financials_balance_sheets
list_financials_cash_flow_statements
list_financials_income_statements
list_financials_ratios
list_short_interest
list_short_volume


In [ ]:
# import json
# from urllib.parse import urlparse, parse_qs
# import pandas as pd

# def list_stocks_floats_compat(client, limit=5000, sort="ticker.asc", max_pages=None):
#     """Compatibility fallback for SDK versions without list_stocks_floats."""
#     path = "/stocks/vX/float"
#     params = {"limit": int(limit), "sort": sort}
#     page = 0

#     while True:
#         resp = client._get(path=path, params=params, raw=True)
#         payload = json.loads(resp.data.decode("utf-8"))

#         for row in payload.get("results", []):
#             yield row

#         next_url = payload.get("next_url")
#         page += 1
#         if not next_url:
#             break
#         if max_pages is not None and page >= max_pages:
#             break

#         # next_url is absolute; convert to relative path + params for client._get
#         parsed = urlparse(next_url)
#         path = parsed.path
#         params = {k: v[0] for k, v in parse_qs(parsed.query).items()}

# # 兼容实现探索：先抓 200 条看看字段和 free_float_percent
# floats_rows = []
# for i, row in enumerate(list_stocks_floats_compat(client, limit=100, sort="ticker.asc", max_pages=2), start=1):
#     floats_rows.append(row)
#     if i >= 200:
#         break

# floats_df = pd.DataFrame(floats_rows)
# print(f"rows: {len(floats_df)}")
# print("columns:", list(floats_df.columns))
# if "free_float_percent" in floats_df.columns:
#     print("free_float_percent nulls:", int(floats_df["free_float_percent"].isna().sum()))
# display(floats_df.head())

rows: 200
columns: ['ticker', 'free_float', 'effective_date', 'free_float_percent']
free_float_percent nulls: 0


,ticker,free_float,effective_date,free_float_percent
0,A,282591673,2026-01-08,99.70
1,AA,258394311,2026-01-29,99.80
2,AABVF,133442193,2026-01-02,83.04
3,AACB,22175000,2026-02-13,100.00
4,AACI,15430468,2025-09-08,65.08


In [ ]:
# 稳态探测：8-K（单线程 + 小分页）
probe_8k = downloader.download_dataset(
    ds="filing_8k_text",
    years=None,
    workers=1,
    limit=1000,
    skip_existing=False,
    delete_empty=True,
    max_pages=None,
    chunk_size=2000,
 )
display(probe_8k.sort_values(["partition"]).reset_index(drop=True))

[filing][filing_8k_text][2024] ok rows=2000


,dataset,category,partition,rows,status,file
0,filing_8k_text,filing,2024,2000,ok,/home/yluel/share/projects/massive_parquet/fil...


In [ ]:
# 诊断：聚焦 aggregate_bars 失败原因
try:
    diag_summary = downloader.download_dataset(
        ds="aggs_daily_market_summary",
        years=[2024],
        workers=1,
        limit=1000,
        skip_existing=False,
        delete_empty=True,
        max_pages=1,
        chunk_size=2000,
    )
    display(diag_summary)
except Exception as e:
    print("diag_error:", repr(e))

In [6]:
# 验证剩余疑难数据集（强制重跑，不依赖 .ok）
verify_datasets = ["corp_dividends", "filing_risk_factors", "news_all", "tickers_all"]
verify_results, verify_summary = downloader.download_all(
    datasets=verify_datasets,
    years=None,
    dataset_workers=1,
    partition_workers=1,
    limit=1000,
    skip_existing=False,
    delete_empty=True,
    max_pages=2,
    chunk_size=5000,
 )
display(verify_summary.sort_values(["dataset", "partition"]).reset_index(drop=True))

corporate_actions:corp_dividends:   0%|          | 0/28 [00:00<?, ?part/s]

[corporate_actions][corp_dividends][2000] ok rows=1
[corporate_actions][corp_dividends][2001] ok rows=2
[corporate_actions][corp_dividends][2002] ok rows=22
[corporate_actions][corp_dividends][2003] ok rows=2000
[corporate_actions][corp_dividends][2004] ok rows=2000
[corporate_actions][corp_dividends][2005] ok rows=2000
[corporate_actions][corp_dividends][2006] ok rows=2000
[corporate_actions][corp_dividends][2007] ok rows=2000
[corporate_actions][corp_dividends][2008] ok rows=2000
[corporate_actions][corp_dividends][2009] ok rows=2000
[corporate_actions][corp_dividends][2010] ok rows=2000
[corporate_actions][corp_dividends][2011] ok rows=2000
[corporate_actions][corp_dividends][2012] ok rows=2000
[corporate_actions][corp_dividends][2013] ok rows=2000
[corporate_actions][corp_dividends][2014] ok rows=2000
[corporate_actions][corp_dividends][2015] ok rows=2000
[corporate_actions][corp_dividends][2016] ok rows=2000
[corporate_actions][corp_dividends][2017] ok rows=2000
[corporate_actions

filing:filing_risk_factors:   0%|          | 0/12 [00:00<?, ?part/s]

[filing][filing_risk_factors][2015] ok rows=2000
[filing][filing_risk_factors][2016] ok rows=2000
[filing][filing_risk_factors][2017] ok rows=2000
[filing][filing_risk_factors][2018] ok rows=2000
[filing][filing_risk_factors][2019] ok rows=2000
[filing][filing_risk_factors][2020] ok rows=2000
[filing][filing_risk_factors][2021] ok rows=2000
[filing][filing_risk_factors][2022] ok rows=2000
[filing][filing_risk_factors][2023] ok rows=2000
[filing][filing_risk_factors][2024] ok rows=2000
[filing][filing_risk_factors][2025] ok rows=2000
[filing][filing_risk_factors][2026] ok rows=2000


news:news_all:   0%|          | 0/1 [00:00<?, ?part/s]

[news][news_all][all] ok rows=2000


tickers:tickers_all:   0%|          | 0/1 [00:00<?, ?part/s]

[tickers][tickers_all][all] ok rows=2000


,dataset,category,partition,rows,status,file,marker
0,corp_dividends,corporate_actions,2000,1,ok,/home/yluel/share/projects/massive_parquet/cor...,/home/yluel/share/projects/massive_parquet/cor...
1,corp_dividends,corporate_actions,2001,2,ok,/home/yluel/share/projects/massive_parquet/cor...,/home/yluel/share/projects/massive_parquet/cor...
2,corp_dividends,corporate_actions,2002,22,ok,/home/yluel/share/projects/massive_parquet/cor...,/home/yluel/share/projects/massive_parquet/cor...
3,corp_dividends,corporate_actions,2003,2000,ok,/home/yluel/share/projects/massive_parquet/cor...,/home/yluel/share/projects/massive_parquet/cor...
4,corp_dividends,corporate_actions,2004,2000,ok,/home/yluel/share/projects/massive_parquet/cor...,/home/yluel/share/projects/massive_parquet/cor...
5,corp_dividends,corporate_actions,2005,2000,ok,/home/yluel/share/projects/massive_parquet/cor...,/home/yluel/share/projects/massive_parquet/cor...
6,corp_dividends,corporate_actions,2006,2000,ok,/home/yluel/share/projects/massive_parquet/cor...,/home/yluel/share/projects/massive_parquet/cor...
7,corp_dividends,corporate_actions,2007,2000,ok,/home/yluel/share/projects/massive_parquet/cor...,/home/yluel/share/projects/massive_parquet/cor...
8,corp_dividends,corporate_actions,2008,2000,ok,/home/yluel/share/projects/massive_parquet/cor...,/home/yluel/share/projects/massive_parquet/cor...
9,corp_dividends,corporate_actions,2009,2000,ok,/home/yluel/share/projects/massive_parquet/cor...,/home/yluel/share/projects/massive_parquet/cor...


In [ ]:
# 启用并下载 8K / 10K（稳态参数）
downloader.DATASETS["filing_10k_sections"].update({
    "mode": "date",
    "partition_freq": "month",
    "date_fields": ["filing_date", "period_end"],
    "sort": "filing_date.desc",
    "page_pause_s": 0.25,
})

for ds in ["filing_8k_text", "filing_10k_sections"]:
    if ds not in datasets_to_download:
        datasets_to_download.append(ds)

filing_8k_10k_results, filing_8k_10k_summary = downloader.download_all(
    datasets=["filing_8k_text", "filing_10k_sections"],
    years=None,
    dataset_workers=1,
    partition_workers=1,
    limit=500,
    skip_existing=True,
    delete_empty=True,
    max_pages=None,
    chunk_size=5000,
 )

display(filing_8k_10k_summary.sort_values(["dataset", "partition"]).reset_index(drop=True))

filing:filing_8k_text:   0%|          | 0/27 [00:00<?, ?part/s]

[retry] path=/stocks/filings/8-K/vX/text attempt=1/12 sleep=6.2s err=HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /stocks/filings/8-K/vX/text?cursor=AggJ-xaaCQn7VJoBBAAAAfv0AQABAQn7Qpo%3D (Caused by ResponseError('too many 502 error responses'))
[retry] path=/stocks/filings/8-K/vX/text attempt=2/12 sleep=6.3s err=HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /stocks/filings/8-K/vX/text?cursor=AggJ-xaaCQn7VJoBBAAAAfv0AQABAQn7Qpo%3D (Caused by ResponseError('too many 502 error responses'))


In [ ]:
# 抗 502 版本：8K / 10K 按月份串行重试下载（避免大分区 cursor 抖动）
import random
import time
import pandas as pd

# 运行时把 10K 改成按日期分片，避免 all 模式单次压力过大
downloader.DATASETS["filing_10k_sections"].update({
    "mode": "date",
    "partition_freq": "month",
    "date_fields": ["filing_date", "period_end"],
    "sort": "filing_date.desc",
    "page_pause_s": 1.2,
})

# 8K 同步放慢分页节奏 + 提高 cursor 场景退避
downloader.DATASETS["filing_8k_text"].update({
    "page_pause_s": 1.2,
})
downloader.request_max_retries = 20
downloader.cursor_retry_floor_s = 12.0

def download_month_with_retry(ds, month_part, attempts=8, base_sleep=6.0):
    last_err = None
    for i in range(1, attempts + 1):
        try:
            # 这里把 month_part 透传给 years 参数，利用 _query 对 YYYY-MM 的月窗口处理
            return downloader.download_dataset(
                ds=ds,
                years=[month_part],
                workers=1,
                limit=1000,
                skip_existing=True,
                delete_empty=True,
                max_pages=None,
                chunk_size=5000,
            )
        except Exception as e:
            last_err = e
            sleep_s = min(180.0, base_sleep * (1.9 ** (i - 1)) + random.uniform(0.0, 2.0))
            print(f"[{ds}][{month_part}] attempt {i}/{attempts} failed: {e}")
            if i < attempts:
                print(f"[{ds}][{month_part}] sleeping {sleep_s:.1f}s and retry...")
                time.sleep(sleep_s)
    raise RuntimeError(f"{ds} month={month_part} failed after {attempts} attempts: {last_err}")

start_year = 2004
end_year = pd.Timestamp.today().year
month_parts = [f"{y}-{m:02d}" for y in range(start_year, end_year + 1) for m in range(1, 13)]

all_rows = []
for ds in ["filing_8k_text", "filing_10k_sections"]:
    print(f"===== start {ds} =====")
    for mp in month_parts:
        m_df = download_month_with_retry(ds, mp, attempts=8, base_sleep=6.0)
        all_rows.append(m_df)
        ok_n = int((m_df["status"] == "ok").sum()) if "status" in m_df.columns else 0
        skipped_n = int((m_df["status"] == "skipped").sum()) if "status" in m_df.columns else 0
        failed_n = int((m_df["status"] == "failed").sum()) if "status" in m_df.columns else 0
        print(f"[{ds}][{mp}] ok={ok_n} skipped={skipped_n} failed={failed_n}")

filing_8k_10k_resilient_summary = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
filing_8k_10k_resilient_summary = filing_8k_10k_resilient_summary.sort_values(["dataset", "partition"]).reset_index(drop=True)
filing_8k_10k_resilient_summary.to_csv("download_results_8k_10k_resilient.csv", index=False)
display(filing_8k_10k_resilient_summary)

===== start filing_8k_text =====


filing:filing_8k_text:   0%|          | 0/1 [00:00<?, ?part/s]

[filing][filing_8k_text][2004-01] empty rows=0
[filing_8k_text][2004-01] ok=0 skipped=0 failed=0


filing:filing_8k_text:   0%|          | 0/1 [00:00<?, ?part/s]

[filing][filing_8k_text][2004-02] empty rows=0
[filing_8k_text][2004-02] ok=0 skipped=0 failed=0


...

[filing][filing_8k_text][2014-10] empty rows=0
[filing_8k_text][2014-10] ok=0 skipped=0 failed=0


filing:filing_8k_text:   0%|          | 0/1 [00:00<?, ?part/s]

[filing][filing_8k_text][2014-11] empty rows=0
[filing_8k_text][2014-11] ok=0 skipped=0 failed=0


filing:filing_8k_text:   0%|          | 0/1 [00:00<?, ?part/s]

[filing][filing_8k_text][2014-12] empty rows=0
[filing_8k_text][2014-12] ok=0 skipped=0 failed=0
